# Notebook 1 — Explore the data

*Days 1–2 · become one with the data.*

In [ ]:
# ======================================================================================
# Setup — run this first (you don't need to read it closely).
# It imports a few libraries, sets a shared plot style, and defines the helper functions used
# throughout the notebook (load_data, the plot helpers, the metric helpers, ...).  These are the
# SAME helpers in every notebook, so your results line up with your teammates'.  If a later cell
# says a helper is "not defined", you probably skipped this cell — run it, then carry on.
# ======================================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------------------
# Paths — resolved so the notebooks work whether they're run from the materials folder or
# from solutions/ .  Outputs and figures always land in the materials root.
# ----------------------------------------------------------------------------------------
def _materials_root():
    """Folder that contains data/ ; searched upward from the working directory."""
    here = os.path.abspath(os.getcwd())
    for d in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]:
        if os.path.isdir(os.path.join(d, "data")):
            return d
    return here

ROOT = _materials_root()
DATA_DIR = os.path.join(ROOT, "data")
OUTPUTS_DIR = os.path.join(ROOT, "outputs")
FIGURES_DIR = os.path.join(ROOT, "figures")
MODELS_DIR = os.path.join(ROOT, "models")     # fitted models saved to disk (experiment tracking)
for _d in (DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)

# Module-level cache of the reconstruction basis (set by load_data; used by reconstruct_spectrum).
_RECON = {"wavelength": None, "basis": None, "is_mock": True, "Xerr": None}
_RAW_CACHE = {}   # path -> full arrays, so repeated load_data() calls don't re-read the file

# Default working-set size.  The real catalog is ~263k stars; we train on a fast deterministic
# subsample by default so in-class fits stay quick AND every notebook sees the SAME stars (which
# keeps the NB2->NB3->NB4 saved-prediction hand-off aligned).  Pass n=None to use the WHOLE
# high-quality catalog (the "does more data help?" / label-transfer-at-scale exercise).
DEFAULT_N = 30000

# Canonical label columns.  The modelling track predicts [Fe/H]; Teff & logg are explored and
# used as the axes for residual / coverage maps.  alpha/age/dist are extra "explore" labels.
LABEL_COLS = ["Teff", "logg", "FeH", "alpha", "age", "dist"]
PRETTY = {"Teff": r"$T_\mathrm{eff}$ [K]", "logg": r"$\log g$ [dex]",
          "FeH": r"[Fe/H] [dex]", "alpha": r"[$\alpha$/M] [dex]",
          "age": "age [Gyr]", "dist": "distance [pc]"}


# ========================================================================================
# Plot style
# ========================================================================================
def set_plot_style():
    """Consistent, readable matplotlib defaults.  Called once in setup."""
    plt.rcParams.update({
        "figure.figsize": (6.4, 4.4), "figure.dpi": 110, "savefig.dpi": 150,
        "savefig.bbox": "tight", "font.size": 12, "axes.titlesize": 13,
        "axes.labelsize": 12, "axes.grid": True, "grid.alpha": 0.25,
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "legend.fontsize": 11, "lines.linewidth": 2.0,
        "scatter.edgecolors": "none", "image.cmap": "viridis",
    })

# A small colour-blind-friendly palette used across the materials for named series.
COLORS = {"linear": "#888888", "knn": "#0072B2", "rf": "#009E73",
          "nn": "#D55E00", "native": "#0072B2", "conformal": "#D55E00",
          "truth": "#222222", "accent": "#CC79A7"}


# ========================================================================================
# Data loading  (the mock generator lives here so the shipped npz and the on-the-fly
# fallback are guaranteed identical)
# ========================================================================================
_C2_NM = 1.438777e7  # second radiation constant hc/k in nm*K


def _build_basis(wave_min=336.0, wave_max=1020.0, n_wave=343, n_bp=55, n_rp=55):
    """Smooth windowed-cosine modes per band, orthonormalised.  Low modes are broad
    (continuum), high modes wiggly (fine structure).  reconstruct = B @ coeff."""
    wave = np.linspace(wave_min, wave_max, n_wave)
    cols = []
    def band(lo, hi, n_modes):
        x = (wave - lo) / (hi - lo)
        win = np.where((x >= 0) & (x <= 1), np.sin(np.clip(x, 0, 1) * np.pi) ** 0.5, 0.0)
        for j in range(n_modes):
            cols.append(np.cos(j * np.pi * np.clip(x, 0, 1)) * win)
    band(wave_min, 690.0, n_bp)
    band(630.0, wave_max, n_rp)
    Q, _ = np.linalg.qr(np.column_stack(cols))
    return wave, Q[:, :n_bp + n_rp]


def _synthesize_mock(n=50000, seed=42):
    """
    # MOCK DATA — swap for the real Zenodo loader.
    Physically-motivated synthetic stand-in for the Laroche & Speagle APOGEE x Gaia-XP
    cross-match.  Returns the same dict that the shipped npz stores.  See the build spec
    and _build/make_mock_dataset.py for the full rationale.  In one sentence: a Planck
    continuum sets Teff; Balmer lines reinforce Teff; gravity-sensitive molecular bands
    carry log g; SHALLOW metal lines (suppressed at high Teff) carry the hard, heteroscedastic
    [Fe/H]; reddening + hidden factors add a realistic irreducible (aleatoric) floor.
    """
    rng = np.random.default_rng(seed)
    wave, B = _build_basis()
    W = wave.size

    # --- labels with Kiel-diagram structure: dwarfs + giants + red clump + metal-poor halo ---
    is_giant = rng.random(n) < 0.45
    Teff = np.empty(n); logg = np.empty(n); FeH = np.empty(n)
    nd, ng = np.count_nonzero(~is_giant), np.count_nonzero(is_giant)
    Td = 4000 + 3100 * rng.beta(1.5, 2.5, nd)
    Teff[~is_giant] = Td
    logg[~is_giant] = 4.62 - 0.00013 * (Td - 4500) + rng.normal(0, 0.10, nd)
    FeH[~is_giant] = rng.normal(-0.05, 0.25, nd)
    Tg = 3800 + 1700 * rng.beta(2.0, 2.3, ng)
    Teff[is_giant] = Tg
    logg[is_giant] = 1.7 + 0.00095 * (Tg - 3800) + rng.normal(0, 0.33, ng)
    FeH[is_giant] = rng.normal(-0.28, 0.34, ng)
    clump = is_giant & (rng.random(n) < 0.33); nc = np.count_nonzero(clump)
    Teff[clump] = rng.normal(4800, 140, nc); logg[clump] = rng.normal(2.45, 0.11, nc)
    FeH[clump] = rng.normal(-0.10, 0.20, nc)
    halo = rng.random(n) < 0.06; nh = np.count_nonzero(halo)
    Teff[halo] = rng.normal(4900, 450, nh)
    logg[halo] = np.clip(rng.normal(2.2, 0.7, nh), 0.5, 3.6)
    FeH[halo] = rng.normal(-1.45, 0.45, nh)
    is_giant = is_giant | halo
    Teff = np.clip(Teff, 3500, 7600); logg = np.clip(logg, 0.3, 5.0); FeH = np.clip(FeH, -2.4, 0.55)
    alpha = np.clip(0.13 - 0.24 * FeH + rng.normal(0, 0.035, n), -0.05, 0.5)
    age = np.clip(2.0 + 6.5 * is_giant - 3.5 * FeH + rng.normal(0, 1.8, n), 0.3, 13.5)
    dist = np.exp(rng.normal(6.7, 0.9, n)) * (1.0 + 1.5 * is_giant)

    # --- forward model: labels -> flux ---
    def gauss(c, w): return np.exp(-0.5 * ((wave - c) / w) ** 2)
    x = _C2_NM / (wave[None, :] * Teff[:, None])
    flux = wave[None, :] ** -5 / np.expm1(np.clip(x, 1e-6, 700))
    flux = flux / flux.mean(axis=1, keepdims=True)
    jump = 1.0 / (1.0 + np.exp((wave - 382.0) / 6.0))
    flux *= 1.0 - (0.18 * np.clip((Teff - 5200) / 2500, 0, 1.4) * (1 + 0.15 * (4.5 - logg)))[:, None] * jump[None, :]
    balmer = sum(a * gauss(c, 9.0) for c, a in [(434., .9), (486.1, 1.1), (656.3, 1.)])
    bstr = np.clip(np.exp((Teff - 5200) / 1500), 0.15, 6.0) * (1 + 0.12 * (4.5 - logg))
    flux *= 1.0 - 0.012 * bstr[:, None] * balmer[None, :]
    g_dwarf = np.clip(logg - 3.0, 0, 2.0)[:, None]; g_giant = np.clip(3.6 - logg, 0, 3.2)[:, None]
    coolf = np.clip((5500 - Teff) / 2000.0, 0.2, 1.4)[:, None]
    grav = np.zeros((n, W))
    for c, a in [(488., 1.), (521., .9), (692., .8)]: grav += a * g_dwarf * gauss(c, 15.)[None, :]
    for c, a in [(421., .9), (793., .8), (883., 1.)]: grav += a * g_giant * gauss(c, 17.)[None, :]
    flux *= 1.0 - 0.038 * coolf * grav
    metal = sum(a * gauss(c, 11.0) for c, a in
                [(422.7, 1.), (460., .7), (517.3, 1.2), (527., .9), (589.3, 1.1),
                 (670.8, .6), (770., .6), (850., .8), (920., .5)])
    # Line depth scales with metal abundance (metal-poor stars have WEAK lines -> little [Fe/H]
    # information -> the dominant, physically-correct source of heteroscedastic [Fe/H] error).
    # Temperature only mildly suppresses lines; gravity gives luminous giants slightly weaker,
    # more variable lines (they are also the sparsest, most-extrapolated regime).
    mstr = 10.0 ** (0.55 * FeH)
    tsupp = np.clip(np.exp(-(Teff - 4500) / 4800.0), 0.6, 1.25)
    line_amp = (mstr * tsupp * np.clip(0.7 + 0.22 * logg, 0.7, 1.4))[:, None]
    flux *= 1.0 - 0.045 * line_amp * metal[None, :]
    ebv = np.abs(rng.normal(0, 0.11, n))[:, None]
    ext = (550.0 / wave) ** 1.0
    flux *= 10.0 ** (-0.4 * 1.6 * ebv * (ext / ext.mean())[None, :])
    P = np.column_stack([np.sin(f * np.pi * (wave - wave[0]) / (wave[-1] - wave[0])) for f in (1.5, 3.5, 6.5)])
    flux *= 1.0 + 0.004 * (rng.normal(0, 1, (n, 3)) @ P.T)
    gflux = np.exp(rng.normal(0.0, 0.8, n))
    snr = np.clip(58 * gflux ** 0.30 * np.exp(rng.normal(0, 0.30, n)), 20, 400)
    obs = gflux[:, None] * np.clip(flux, 1e-3, None)
    noise_std = obs.mean(axis=1, keepdims=True) / snr[:, None]        # photon-noise std (per star)
    obs = obs + rng.normal(0, 1, obs.shape) * noise_std
    X = (obs @ B).astype(np.float32)
    # per-coefficient measurement error on the RAW coefficients (the orthonormal basis preserves the
    # per-component noise std) — used by the NB3 input-Monte-Carlo stretch.
    Xerr = np.repeat(noise_std, 110, axis=1).astype(np.float32)
    return dict(X=X, Xerr=Xerr, Teff=Teff, logg=logg, FeH=FeH, alpha=alpha, age=age, dist=dist,
                snr=snr, gflux=gflux, source_id=(1_000_000_000 + np.arange(n)),
                wavelength=wave, basis=B.astype(np.float32), is_mock=True)


def load_data(path=None, n=DEFAULT_N, seed=0, verbose=True):
    """
    Load the dataset in one line:  X, y, meta = load_data()

    Parameters
    ----------
    n    : working-set size.  Returns a deterministic random subsample of `n` stars (default
           DEFAULT_N=30000) so fits are fast and every notebook sees the same stars.  Pass
           `n=None` to use the ENTIRE high-quality catalog (the scale-up exercise) — slower.
    seed : seed for the subsample (keep it fixed so the NB2->NB3->NB4 hand-off stays aligned).

    Returns
    -------
    X    : float array, shape (n, 110) — the Gaia XP coefficients (55 BP + 55 RP).
    y    : DataFrame with columns Teff, logg, FeH (+ alpha, dist) — the labels.
    meta : DataFrame with source_id, snr, gflux — per-star bookkeeping (gflux is the
           brightness scale used by normalize_by_g).
    """
    if path is None:
        for cand in ("xp_apogee_real.npz", "xp_apogee_mock.npz"):
            p = os.path.join(DATA_DIR, cand)
            if os.path.exists(p):
                path = p
                break
    key = path if path is not None else "__synth__"
    if key in _RAW_CACHE:
        data = _RAW_CACHE[key]
    elif path is not None and os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        data = {k: d[k] for k in d.files}
        _RAW_CACHE[key] = data
    else:
        if verbose:
            print("No data file found — synthesising the mock dataset on the fly.")
        data = _synthesize_mock()
        _RAW_CACHE[key] = data
    is_mock = bool(data.get("is_mock", True))
    src = os.path.basename(path) if path else "synthesised mock"

    _RECON["wavelength"] = np.asarray(data["wavelength"], float)
    _RECON["basis"] = np.asarray(data["basis"], float)
    _RECON["is_mock"] = is_mock

    X_all = np.asarray(data["X"], dtype=float)
    N = X_all.shape[0]
    if n is not None and n < N:
        sel = np.sort(np.random.default_rng(seed).permutation(N)[:n])
    else:
        sel = np.arange(N)
    X = X_all[sel]
    _RECON["Xerr"] = np.asarray(data["Xerr"], float)[sel] if "Xerr" in data else None  # for input-MC (NB3)
    y = pd.DataFrame({c: np.asarray(data[c], float)[sel] for c in LABEL_COLS if c in data})
    meta = pd.DataFrame({k: np.asarray(data[k]).ravel()[sel] for k in ("source_id", "snr", "gflux") if k in data})
    if verbose:
        tag = "MOCK (synthetic)" if is_mock else "REAL (Gaia XP x APOGEE)"
        extra = f" (subsampled from {N:,}; pass n=None for all)" if len(sel) < N else ""
        print(f"Loaded {src}: {tag}  —  {X.shape[0]:,} stars{extra}, {X.shape[1]} coefficients.")
    return X, y, meta


# ========================================================================================
# Spectrum reconstruction
# ========================================================================================
def load_input_errors():
    """Per-coefficient measurement errors for the stars from the LAST `load_data()` call (same rows,
    same order, same (n, 110) shape as the `X` you just loaded) — on the RAW coefficient scale.
    For the NB3 input-Monte-Carlo stretch:  X_perturbed = X + rng.normal(0, load_input_errors()).
    Returns None if the dataset has no stored errors (call `load_data()` first)."""
    if _RECON["Xerr"] is None:
        load_data(verbose=False)
    return _RECON["Xerr"]


def reconstruct_spectrum(coeffs):
    """
    Turn a star's 110 XP coefficients into a sampled spectrum: returns (wavelength_nm, flux).
    Accepts a single (110,) vector or an (M, 110) batch.

    On the shipped data this uses the stored smooth basis (B @ coeff) — no extra packages.

    # REAL DATA: with the genuine Gaia archive continuous representation you would instead
    # reconstruct with GaiaXPy, e.g.:
    #     from gaiaxpy import calibrate
    #     sampling = np.arange(336, 1021, 2.0)              # nm
    #     spectra, sampled_wl = calibrate(source_dataframe, sampling=sampling, save_file=False)
    # The stored basis is recovered from Zenodo's wavelength-space spectra at curation time
    # (see _build/prepare_real_data.py), so this helper stays identical for mock and real.
    """
    if _RECON["basis"] is None:
        load_data(verbose=False)
    B, wave = _RECON["basis"], _RECON["wavelength"]
    coeffs = np.asarray(coeffs, float)
    flux = coeffs @ B.T
    return wave, flux


# ========================================================================================
# Preprocessing recipe:  normalize by the G-band  ->  standardize (fit on train only)
# ========================================================================================
def normalize_by_g(X, g):
    """Divide each star's coefficients by its G-band brightness scale `g` (a column of meta).
    Removes brightness/distance and keeps the spectral SHAPE — what carries the labels."""
    g = np.asarray(g, float).reshape(-1, 1)
    return np.asarray(X, float) / g


def standardize(X_train, X_other):
    """Z-score the coefficients using statistics from the TRAINING set only, then apply the
    same shift/scale to another split.  Returns (X_train_scaled, X_other_scaled).
    (Equivalent to sklearn's StandardScaler fit on train; spelled out for clarity.)"""
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return (X_train - mu) / sd, (X_other - mu) / sd


# ========================================================================================
# Splitting  (fixed seed so every student gets the same train / cal / test stars)
# ========================================================================================
def train_cal_test_split(X, y, seed=0, fracs=(0.6, 0.2, 0.2)):
    """Reproducible 60/20/20 split into train / calibration / test.
    Returns a dict with X_* arrays, y_* DataFrames, and idx_* index arrays.
    Calibration is held out for the uncertainty work in NB3/NB4."""
    X = np.asarray(X, float)
    y = y.reset_index(drop=True)
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_tr = int(fracs[0] * n); n_ca = int(fracs[1] * n)
    i_tr, i_ca, i_te = idx[:n_tr], idx[n_tr:n_tr + n_ca], idx[n_tr + n_ca:]
    return {
        "X_train": X[i_tr], "X_cal": X[i_ca], "X_test": X[i_te],
        "y_train": y.iloc[i_tr].reset_index(drop=True),
        "y_cal": y.iloc[i_ca].reset_index(drop=True),
        "y_test": y.iloc[i_te].reset_index(drop=True),
        "idx_train": i_tr, "idx_cal": i_ca, "idx_test": i_te,
    }


# ========================================================================================
# Models — the four estimators the students compare.  Defined ONCE here so NB2, NB3 and NB4
# build byte-identical models, which is what lets predictions and intervals line up across
# notebooks and across the three students.  Each is a Pipeline(StandardScaler + estimator),
# so you feed it the G-normalized coefficients and it standardizes internally (fit on train).
# ========================================================================================
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_NAMES = {"linear": "Linear regression (baseline)", "knn": "K-nearest neighbors",
               "rf": "Random forest", "nn": "Neural network (MLP)"}

def make_model(name, **overrides):
    """Canonical Pipeline(StandardScaler + estimator) for a model name in {linear,knn,rf,nn}.
    Pass overrides to tune one knob, e.g. make_model('knn', n_neighbors=20).  The step name for
    the estimator is its lowercased class name (e.g. 'kneighborsregressor') — handy in NB3 when
    you reach inside the fitted pipeline for a model-native uncertainty."""
    if name == "linear":
        est = LinearRegression(**overrides)
    elif name == "knn":
        est = KNeighborsRegressor(**{"n_neighbors": 12, **overrides})
    elif name == "rf":
        est = RandomForestRegressor(**{"n_estimators": 150, "min_samples_leaf": 3,
                                       "random_state": 0, "n_jobs": -1, **overrides})
    elif name == "nn":
        est = MLPRegressor(**{"hidden_layer_sizes": (64, 64), "alpha": 1e-3, "max_iter": 500,
                              "early_stopping": True, "random_state": 0, **overrides})
    else:
        raise ValueError(f"unknown model '{name}' (use one of {list(MODEL_NAMES)})")
    return make_pipeline(StandardScaler(), est)


# ========================================================================================
# Metrics
# ========================================================================================
def regression_metrics(y_true, y_pred):
    """Return a dict of RMSE, MAE, bias (mean signed error), and R^2."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_pred - y_true
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2)) or 1.0
    return {"RMSE": float(np.sqrt(np.mean(err ** 2))), "MAE": float(np.mean(np.abs(err))),
            "bias": float(np.mean(err)), "R2": float(1.0 - ss_res / ss_tot)}


# ========================================================================================
# Plots — predictions & residuals
# ========================================================================================
def plot_pred_vs_true(y_true, y_pred, title=None, ax=None, color=None, label=None, units=""):
    """Scatter of predicted vs true with the 1:1 line and an RMSE annotation."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    if ax is None:
        _, ax = plt.subplots()
    lo = min(y_true.min(), y_pred.min()); hi = max(y_true.max(), y_pred.max())
    pad = 0.04 * (hi - lo + 1e-9)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "--", color="0.4", lw=1.3, zorder=1)
    ax.scatter(y_true, y_pred, s=8, alpha=0.35, color=color or COLORS["rf"], label=label, zorder=2)
    m = regression_metrics(y_true, y_pred)
    ax.text(0.04, 0.96, f"RMSE = {m['RMSE']:.3g}{units}\nbias = {m['bias']:+.2g}{units}",
            transform=ax.transAxes, va="top", ha="left", fontsize=10,
            bbox=dict(boxstyle="round", fc="white", ec="0.8", alpha=0.85))
    ax.set_xlabel(f"true{(' ' + units) if units else ''}")
    ax.set_ylabel(f"predicted{(' ' + units) if units else ''}")
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect("equal", "box")
    if title:
        ax.set_title(title)
    if label:
        ax.legend(loc="lower right")
    return ax


def plot_residuals(y_true, y_pred, feature=None, feature_name="value", ax=None, color=None):
    """Residuals (pred - true) vs the truth, or vs an external `feature` (e.g. Teff).
    A flat band around zero means well-behaved errors; trends/fans flag structure."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    resid = y_pred - y_true
    xx = np.asarray(feature, float) if feature is not None else y_true
    xlabel = feature_name if feature is not None else "true value"
    if ax is None:
        _, ax = plt.subplots()
    ax.axhline(0.0, color="0.4", ls="--", lw=1.3, zorder=1)
    ax.scatter(xx, resid, s=8, alpha=0.35, color=color or COLORS["rf"], zorder=2)
    ax.set_xlabel(xlabel); ax.set_ylabel("residual (pred − true)")
    return ax


# ========================================================================================
# Uncertainty — coverage, reliability, comparison
# ========================================================================================
def empirical_coverage(y_true, lower, upper):
    """Fraction of true values that fall inside [lower, upper]."""
    y_true = np.asarray(y_true, float)
    return float(np.mean((y_true >= np.asarray(lower, float)) & (y_true <= np.asarray(upper, float))))


def plot_reliability(y_true, lower, upper, target=None, feature=None, feature_name="bin",
                     n_bins=8, ax=None, color=None, label=None):
    """
    'Are the error bars honest, everywhere?'  Bins the points (by an external `feature` such
    as Teff if given, else by the interval centre) and plots empirical coverage per bin against
    the `target` line.  Bars sitting below the target reveal where intervals under-cover.
    """
    y_true = np.asarray(y_true, float)
    lower = np.asarray(lower, float); upper = np.asarray(upper, float)
    inside = (y_true >= lower) & (y_true <= upper)
    xx = np.asarray(feature, float) if feature is not None else 0.5 * (lower + upper)
    if ax is None:
        _, ax = plt.subplots()
    edges = np.quantile(xx, np.linspace(0, 1, n_bins + 1))
    edges[-1] += 1e-9
    centers, cov = [], []
    for i in range(n_bins):
        m = (xx >= edges[i]) & (xx < edges[i + 1])
        if m.sum() >= 5:
            centers.append(xx[m].mean()); cov.append(inside[m].mean())
    ax.plot(centers, cov, "o-", color=color or COLORS["native"], label=label)
    if target is not None:
        ax.axhline(target, color="0.4", ls="--", lw=1.3, label=f"target = {target:.0%}")
    ax.set_xlabel(feature_name if feature is not None else "interval centre")
    ax.set_ylabel("empirical coverage")
    ax.set_ylim(0, 1.02)
    if target is not None or label:
        ax.legend(loc="lower center")
    return ax


def compare_intervals(y_true, intervals, target=None, ax=None):
    """
    Compare several named interval sets.  `intervals` maps name -> (lower, upper).
    Draws two bars per method — coverage and median width — and returns a tidy summary table.
    """
    y_true = np.asarray(y_true, float)
    rows = []
    for name, (lo, hi) in intervals.items():
        rows.append({"method": name, "coverage": empirical_coverage(y_true, lo, hi),
                     "median_width": float(np.median(np.asarray(hi, float) - np.asarray(lo, float)))})
    tbl = pd.DataFrame(rows).set_index("method")
    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(9.5, 4.0))
    names = list(tbl.index)
    cols = [COLORS.get(n.split()[0].lower(), COLORS["accent"]) for n in names]
    ax[0].bar(names, tbl["coverage"], color=cols)
    if target is not None:
        ax[0].axhline(target, color="0.4", ls="--", lw=1.3, label=f"target = {target:.0%}")
        ax[0].legend()
    ax[0].set_ylabel("empirical coverage"); ax[0].set_ylim(0, 1.02); ax[0].set_title("Coverage")
    ax[1].bar(names, tbl["median_width"], color=cols)
    ax[1].set_ylabel("median interval width"); ax[1].set_title("Sharpness (narrower is better)")
    for a in ax:
        a.tick_params(axis="x", rotation=20)
    return tbl


# ========================================================================================
# Small convenience used in worked examples
# ========================================================================================
def savefig(name, fig=None):
    """Save a figure into figures/ for reuse on the poster.  Returns the path."""
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt.gcf()).savefig(path)
    return path


def save_outputs(filename, **arrays):
    """Save named arrays into outputs/<filename> (the shared NB2->NB3->NB4 contract)."""
    path = os.path.join(OUTPUTS_DIR, filename)
    np.savez(path, **arrays)
    return path


def load_outputs(filename):
    """Load an outputs/ npz written by an earlier notebook; returns a dict of arrays."""
    d = np.load(os.path.join(OUTPUTS_DIR, filename), allow_pickle=True)
    return {k: d[k] for k in d.files}


# ----- Experiment tracking: save fitted models + log runs (so you can reproduce & compare) -----
def save_model(model, name):
    """Save a fitted model/pipeline to models/<name>.joblib. Returns the path.
    The point: a tuned model is an experiment result — persist it so NB3 can load the exact model
    you tuned (no silent refit), and so you can reproduce a run weeks later."""
    import joblib
    path = os.path.join(MODELS_DIR, f"{name}.joblib")
    joblib.dump(model, path)
    return path


def load_model(name):
    """Load a model previously saved with save_model(...)."""
    import joblib
    return joblib.load(os.path.join(MODELS_DIR, f"{name}.joblib"))


def log_experiment(name, params, metrics, filename=None):
    """Append one experiment run (its hyper-parameters + its metrics) to a CSV and return the full
    log as a DataFrame. This is the bare-bones version of what tools like MLflow / Weights & Biases
    do: a durable, comparable record of 'what did I try, and how did it score?'.
      params  : dict of hyper-parameters, e.g. {'n_neighbors': 12, 'weights': 'distance'}
      metrics : dict of scores, e.g. {'cv_rmse': 0.21, 'test_rmse': 0.20}
    """
    path = os.path.join(OUTPUTS_DIR, filename or f"{name}_experiments.csv")
    row = {"run": 1, **{f"param_{k}": v for k, v in params.items()}, **metrics}
    if os.path.exists(path):
        prev = pd.read_csv(path)
        row["run"] = int(prev["run"].max()) + 1 if "run" in prev else len(prev) + 1
        # Rebuild from records rather than pd.concat: a run that logs different params/metrics than
        # an earlier one would otherwise make pandas align mismatched columns into all-NA entries,
        # which raises a (noisy, harmless) FutureWarning.  Building one DataFrame from a list of
        # dicts fills the gaps with NaN silently and preserves column order.
        log = pd.DataFrame(prev.to_dict("records") + [row])
    else:
        log = pd.DataFrame([row])
    log.to_csv(path, index=False)
    return log

set_plot_style()
print('Setup complete — helpers ready.')


## 1 · Welcome — today we become *one with the data*

**No modeling on Day 1.** Before we predict anything, we build **intuition**: we look at the
**inputs**, look at the **labels** we will eventually predict, and learn what makes some labels easy
and others hard. This matters *especially because the dataset is huge* — about **263,000 stars**, each
described by **110 numbers**. (We work with a fast, fixed **30,000-star slice** so every cell runs in
seconds; `load_data(n=None)` would pull the whole catalog.) At that scale you cannot eyeball rows in a
spreadsheet — you have to *plot* your way to understanding. That is the entire job today.

A bit of vocabulary we'll use constantly:
- A **label** is a physical number describing a star that we want to predict — its temperature, its
  gravity, its chemical makeup.
- A **feature** (or **input**) is a number the model gets to read. Here each star is described by
  **110 numbers** (a compressed version of its spectrum); those 110 numbers are our features.

So the whole project is: *110 numbers in → a stellar label out.*

**The four-part rhythm.** Most sections move through the same beats:
- **Explain** — read a short bit of background (cells like this one), always with a reference to dig deeper.
- **Worked** — run a cell and watch it work; nothing for you to change.
- **Your turn** — a cell that already *runs as-is* but leaves the real plot for you. Look for
  `# <- your turn:`; fill in the marked line, re-run, and write one sentence on what you see.
- **Explore** — open-ended prompts with no single right answer.

If a cell errors, **grab a TA** — don't fight it alone.

**Team note.** Everyone runs the whole notebook, but each of you *leads* part of the exploration and
presents it at the debrief:
- **Student A** leads: example-spectra gallery + signal-to-noise / error structure.
- **Student B** leads: coefficient scales + the feature–label correlation heatmap.
- **Student C** leads: the label corner plot + the Kiel diagram.

Minimum deliverable today: **2–3 annotated plots each**.

> *Where this data comes from:* the Gaia mission's low-resolution spectra cross-matched with precise
> APOGEE labels, following Laroche & Speagle (2025, [arXiv:2404.07316](https://arxiv.org/abs/2404.07316)).
> The spectra are reconstructed the way [GaiaXPy](https://gaiaxpy.readthedocs.io) does it. Full story
> in §2.

*One housekeeping cell first.* The Setup cell above already defined every helper. Here we pull in a
few extra scikit-learn pieces we'll use today, and **safely** check for two optional packages
(`umap` for a nonlinear map, `statsmodels` for regression statistics). The `try/except` means the
notebook still runs even if one is missing — it just skips the optional bit.

In [ ]:
# Extra imports for today (all pre-installed; we never pip install in these notebooks).
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

# Optional extras — guarded so a missing package never breaks the notebook.
try:
    import umap                       # nonlinear 2-D embedding (the §6h stretch)
    HAVE_UMAP = True
except Exception:
    HAVE_UMAP = False

try:
    import statsmodels.api as sm      # OLS with p-values / confidence intervals (the §9c extra)
    HAVE_SM = True
except Exception:
    HAVE_SM = False

try:
    import shap                       # model-agnostic feature attributions (the §9c cross-check)
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False

print("HAVE_UMAP =", HAVE_UMAP, " | HAVE_SM =", HAVE_SM, " | HAVE_SHAP =", HAVE_SHAP)

## 2 · What are we working with?

**A spectrum** is a star's light split up by wavelength — the way a prism spreads sunlight into a
rainbow. Its smooth overall *shape* tells you roughly how hot the star is (hotter stars look bluer,
cooler stars redder). Stamped on top of that shape are dark **absorption lines**: narrow dips where
atoms and molecules in the star's outer layers soak up specific wavelengths. The pattern and depth of
those dips is a fingerprint of the star's temperature, gravity, and composition.

**The three labels** astronomers read off that fingerprint (and that we'll work with):

- **Effective temperature, $T_\mathrm{eff}$** (in kelvin, K): the surface temperature. Our stars run
  from a few thousand K up to ~6,500 K (the Sun is ≈ 5,772 K). $T_\mathrm{eff}$ sets the spectrum's
  overall shape, so it is the **easiest** label to read.
- **Surface gravity, $\log g$** (in **dex** — one dex is a factor of 10 on a base-10 log scale): how
  compact the star is. A dense **dwarf** like the Sun has $\log g \approx 4.4$; a puffed-up **giant**
  has $\log g \approx 1$–$3$. Same mass, very different size. **Medium** difficulty.
- **Metallicity, [Fe/H]** (dex): the star's heavy-element content compared to the Sun. To astronomers,
  **"metals"** means *everything heavier than hydrogen and helium*. [Fe/H] = 0 is solar; −1 means
  one-tenth solar (metal-poor, usually old). Metals leave only *shallow* marks at low resolution, so
  [Fe/H] is the **hardest** label — **and it's our main prediction target**, precisely because being
  hard is what makes honest uncertainty matter.

**The inputs — Gaia XP spectra.** Gaia is a European Space Agency satellite that surveyed about two
billion stars and released low-resolution spectra from two onboard prisms: the **B**lue **P**hotometer
(**BP**) and the **R**ed **P**hotometer (**RP**), together nicknamed **"XP"**. To save space, Gaia
ships each spectrum *compressed* into **110 coefficients** (55 BP + 55 RP): multiply them back through
a fixed set of template shapes (a **basis**) and you recover the curve. Those 110 numbers are our
features.

**The answers — APOGEE labels.** APOGEE is a high-resolution survey that takes sharp spectra and
measures precise $T_\mathrm{eff}$, $\log g$, and [Fe/H]. **Laroche & Speagle cross-matched** APOGEE's
precise labels with Gaia's cheap XP spectra (a **cross-match** simply pairs up stars that both surveys
observed). That overlap is our training set: 110 numbers in, labels out — a **supervised learning**
problem (learning from examples where we know the answer).

**Why it matters — label transfer.** Precise labels are scarce; XP spectra exist for hundreds of
millions of stars. Learn the map on the overlap, then *transfer* the labels by applying it to the rest
of the sky → map the chemistry of the whole Milky Way. That is a real, active research problem, and
this project is a scaled-down version of it.

*Sources:* the paper ([arXiv:2404.07316](https://arxiv.org/abs/2404.07316) /
[ADS](https://ui.adsabs.harvard.edu/abs/2025ApJ...979....5L)); Gaia BP/RP
([ESA overview](https://www.cosmos.esa.int/web/gaia/iow_20220131)); APOGEE/ASPCAP
([SDSS](https://www.sdss4.org/dr17/irspec/)). We'll rebuild the astronomer's
[Kiel / H–R diagram](https://en.wikipedia.org/wiki/Hertzsprung%E2%80%93Russell_diagram) straight from
this data in §6.

## 3 · Load the data and take a first look  *(Worked)*

Let's load the cross-match in one line. `load_data()` returns three things:
- `X` — the 110-coefficient inputs, one row per star;
- `y` — a table of labels ($T_\mathrm{eff}$, $\log g$, [Fe/H], and a couple of extras);
- `meta` — bookkeeping: an id, a **signal-to-noise** (S/N) measure, and `gflux`, a brightness scale
  we'll use to preprocess.

**Signal-to-noise (S/N)** is how strong the real signal is compared to the random measurement noise —
higher means a cleaner spectrum. The first time you run it, `load_data()` prints what it loaded —
**watch the printout**: it reports both the size of the slice you got *and* the size of the full
catalog it came from.

In [ ]:
X, y, meta = load_data()                  # one line; prints what it loaded (note the catalog size!)
print("X (inputs):", X.shape, X.dtype)    # expect (N, 110): N stars, 110 coefficients each
print("labels available:", list(y.columns))
print("meta columns:    ", list(meta.columns))

display(y.describe().round(2))            # ranges / typical values for every label
display(meta.head(3))                     # source_id, snr, gflux for the first 3 stars

*Each star is **110 input numbers** plus a row of labels. Note the printed catalog size: our 30k
rows are a fast slice of a **far larger** catalog (~263k high-quality stars) — that scale is exactly
why we lean on plots and summary statistics instead of reading rows. Note too the label ranges:
$T_\mathrm{eff}$ in the thousands of K, $\log g$ and [Fe/H] in dex. Those ranges are the world our
model has to work in.*

## 4 · Seeing a spectrum: from 110 numbers back to a curve  *(Worked)*

The 110 numbers *are* the spectrum, just compressed. `reconstruct_spectrum(coeffs)` multiplies them
back through the fixed basis and returns `(wavelength_nm, flux)`, so we can actually *see* the curve.
Let's plot the coolest and the hottest star in the sample and compare their shapes.

> *Under the hood:* on the shipped data this uses a stored smooth basis (no extra packages); on the
> genuine Gaia archive the same helper would call **GaiaXPy** — the plot looks the same to you.

In [ ]:
# Pick the coolest and hottest star in the sample to contrast spectral SHAPE.
i_cool = int(y["Teff"].idxmin())
i_hot  = int(y["Teff"].idxmax())

wave, flux_cool = reconstruct_spectrum(X[i_cool])   # wave and flux are 1-D arrays of equal length
_,    flux_hot  = reconstruct_spectrum(X[i_hot])

fig, ax = plt.subplots()
ax.plot(wave, flux_cool, label=f"cool star  (Teff ≈ {y['Teff'][i_cool]:.0f} K)", color=COLORS["knn"])
ax.plot(wave, flux_hot,  label=f"hot star   (Teff ≈ {y['Teff'][i_hot]:.0f} K)", color=COLORS["nn"])
ax.axvspan(336, 680, alpha=0.05, color="blue")      # BP region (rough)
ax.axvspan(680, 1020, alpha=0.05, color="red")      # RP region (rough)
ax.set_xlabel("wavelength [nm]"); ax.set_ylabel("flux (arbitrary units)")
ax.set_title("Two reconstructed Gaia XP spectra")
ax.legend()
plt.show()

*The two stars have clearly different overall shapes — the cool star carries relatively more flux at
long (red) wavelengths, the hot star at short (blue) wavelengths. That overall slope is mostly
$T_\mathrm{eff}$. The subtler dips carry $\log g$ and [Fe/H] — and those are much harder to see by eye,
which is the whole point of the project.*

## 5 · Two very different "normalizations"

Before any plotting or modeling, raw coefficients get cleaned up in **two conceptually different**
ways. The instructor wants you to keep these straight, because students constantly blur them together:

- **§5a — a PHYSICS normalization** (`normalize_by_g`): astrophysically motivated. It divides out a
  star's brightness/distance so that only the spectral **shape** — the part that carries the labels —
  remains.
- **§5b — a GENERIC standardization** (per-feature z-score / `StandardScaler`): a purely **numerical**
  convenience. It puts all 110 coefficients on comparable scales so distance- and gradient-based
  methods aren't hijacked by a few large columns. It is *not* physically motivated.

One reshapes the *physics*; the other reshapes the *numbers*. We'll do each, look at the before/after,
and say in one line which is which.

*Reference:* scikit-learn's [preprocessing user guide](https://scikit-learn.org/stable/modules/preprocessing.html)
covers standardization (and many other generic scalers) in detail.

### 5a · Physics normalization — divide out brightness  *(Worked)*

Two identical stars at different distances look different in *brightness* but have the **same spectral
shape**, and shape is what encodes $T_\mathrm{eff}$, $\log g$, [Fe/H]. `normalize_by_g(X, meta["gflux"])`
divides each star's coefficients by its G-band brightness scale `gflux` (Gaia's broad optical
brightness). After this, two stars that differ only in distance line up — the nuisance is gone, the
physics is kept. Let's watch it happen on one reconstructed spectrum.

In [ ]:
# Take one star; reconstruct its spectrum BEFORE and AFTER dividing out brightness.
i = int(y["Teff"].idxmin())                          # any star will do
Xn = normalize_by_g(X, meta["gflux"])                # physics step: same shape (N, 110)

wave, flux_raw  = reconstruct_spectrum(X[i])         # raw coefficients -> flux
_,    flux_norm = reconstruct_spectrum(Xn[i])        # brightness-normalized -> flux

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(wave, flux_raw, color=COLORS["knn"])
ax[0].set_title("before: raw (brightness-scaled)")
ax[0].set_xlabel("wavelength [nm]"); ax[0].set_ylabel("flux (arb.)")
ax[1].plot(wave, flux_norm, color=COLORS["rf"])
ax[1].set_title("after: divided by G-band brightness")
ax[1].set_xlabel("wavelength [nm]"); ax[1].set_ylabel("flux / G  (arb.)")
plt.tight_layout(); plt.show()

print(f"this star's gflux (brightness scale) = {meta['gflux'][i]:.3g}")
print(f"peak flux  before: {flux_raw.max():.3g}   after: {flux_norm.max():.3g}  "
      f"(the SHAPE is unchanged; only the overall scale moved)")

*The two curves have the **same shape** — the only thing that changed is the vertical scale. That is
the whole point: brightness (which depends on distance, an accident of where the star happens to be)
is divided out, and the shape that actually carries the labels is preserved. This step is **physics**,
and there's no off-the-shelf scikit-learn tool for it — it's specific to spectra.*

### 5b · Generic standardization — put every coefficient on the same footing  *(Worked)*

Even after the physics step, the 110 coefficients live on **wildly different numerical scales** (some
are thousands of times larger than others). Any method that measures *distances* between stars
(like K-nearest neighbors) or follows *gradients* (like a neural net) would then be dominated by a
handful of big columns. **Standardizing** fixes this: for each coefficient, subtract its mean and
divide by its **standard deviation** (a measure of spread), so every column ends up centered at 0 with
spread 1 — a **z-score**. scikit-learn calls this `StandardScaler`; the helper `standardize(...)` does
the same thing, fitting the mean/std on the **training** data only (the no-leakage rule you'll formalize
in NB2). This is a *numerical* convenience, **not** physics.

In [ ]:
# Standardize the (already brightness-normalized) coefficients, fitting stats on TRAIN only.
sp = train_cal_test_split(Xn, y, seed=0)             # fixed seed -> everyone gets the same split
Xtr_s, Xte_s = standardize(sp["X_train"], sp["X_test"])

# Before/after on the coefficient SCALES (per-column standard deviation):
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(sp["X_train"].std(axis=0), color=COLORS["rf"]); ax[0].set_yscale("log")
ax[0].set_title("before standardizing"); ax[0].set_xlabel("coefficient index"); ax[0].set_ylabel("column std (log)")
ax[1].plot(Xtr_s.std(axis=0), color=COLORS["rf"]); ax[1].set_ylim(0, 1.4)
ax[1].set_title("after standardizing"); ax[1].set_xlabel("coefficient index"); ax[1].set_ylabel("column std")
plt.tight_layout(); plt.show()

print("before: column std ranges from %.2g to %.2g  (a factor of ~%.0f)"
      % (sp["X_train"].std(0).min(), sp["X_train"].std(0).max(),
         sp["X_train"].std(0).max() / sp["X_train"].std(0).min()))
print("after:  every column std is ~1.0  (mean %.3f)" % Xtr_s.std(0).mean())

*Before, the coefficients span a thousand-fold-plus range in scale (the left panel is on a log axis)
— a few of them would dominate any distance or gradient a model computes. After standardizing, all 110
sit at std ≈ 1, so every coefficient gets a fair say. To recap the distinction: **§5a (divide by G) is
physics** — it removes a real astrophysical nuisance; **§5b (z-score) is bookkeeping** — it just makes
the numbers play nicely with our algorithms.*

## 6 · The active EDA menu — *you build these*

Now the fun part: *look* at the data from every angle. The menu below is a **recommendation**, not a
checklist — the real goal is to build intuition for **what is predictive** *before* we fit a single
model. Correlations, error structure, redundancy, and dimensionality all matter at this scale.

The **first item (6a) is fully worked** as a template. The **rest are "Your turn"**: each one prepares
the data and sets up a labelled figure, then leaves the actual plotting line for you, marked
`# <- your turn:`. An empty labelled plot still *runs*, so the whole notebook executes top to bottom —
your job is to fill in the marked line, re-run, and write one sentence on what you see.

**Who leads each item** (everyone runs everything):
- **Student A:** 6a example spectra · 6b S/N & error structure.
- **Student B:** 6c coefficient scales · 6d feature–label correlation heatmap.
- **Student C:** 6e label corner · 6f Kiel diagram.
- **Everyone:** 6g PCA (recommended) · 6h UMAP (stretch).

If time is short, do your own two thoroughly and skim the rest. Bring **2–3 annotated plots** to the
debrief.

### 6a · Example spectra across temperature  *(Worked exemplar — Student A leads)*

In [ ]:
# GOAL: see how spectral SHAPE changes with temperature. (This one is fully WORKED — your template.)
order = np.argsort(y["Teff"].to_numpy())
picks = order[np.linspace(0, len(order) - 1, 6).astype(int)]   # 6 stars, coolest -> hottest

fig, ax = plt.subplots()
for i in picks:
    w, f = reconstruct_spectrum(X[i])
    ax.plot(w, f, label=f"{y['Teff'][i]:.0f} K", alpha=0.9)
ax.set_xlabel("wavelength [nm]"); ax.set_ylabel("flux (arb.)")
ax.set_title("Example spectra across temperature")
ax.legend(title="Teff", ncol=2, fontsize=9)
plt.show()

# This cell is your TEMPLATE for the build-its below: prepare data -> set up a labelled figure -> plot.
# (Optional) your turn: sort by "FeH" instead of "Teff" above. Does the SHAPE reorder as cleanly?
#   (Hint: Teff dominates shape; FeH is subtle -> that's why [Fe/H] is the hard label.)

*Temperature visibly reorders the spectra into a clean fan; metallicity barely budges them. That's
the central difficulty of this project in one picture — and the layout of this cell (prepare data →
labelled figure → plot) is the **template** for every build-it below.*

### 6b · Signal-to-noise & error structure  *(Your turn — Student A leads)*

In [ ]:
# GOAL: understand where the data is clean vs noisy. Higher S/N = cleaner spectrum.
# Idea: does data quality track brightness or temperature? (relevant later when a model struggles)
snr       = meta["snr"].to_numpy()
logbright = np.log10(meta["gflux"].to_numpy())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Left panel is set up for a HISTOGRAM of S/N:
ax[0].set_xlabel("S/N"); ax[0].set_ylabel("number of stars"); ax[0].set_title("S/N distribution")
# <- your turn (6b-i): histogram the S/N -> ax[0].hist(snr, bins=40, color=COLORS["knn"])

# Right panel is set up for a SCATTER of S/N vs (log) brightness:
ax[1].set_xlabel(r"$\log_{10}$ G-band brightness"); ax[1].set_ylabel("S/N")
ax[1].set_title("S/N vs brightness")
# <- your turn (6b-ii): scatter S/N against brightness -> ax[1].scatter(logbright, snr, s=6, alpha=0.3)
#    Then try the SAME scatter against temperature (y["Teff"]) instead. Which explains S/N better?

plt.tight_layout(); plt.show()

print(f"S/N: median {np.median(snr):.2g}, range {snr.min():.2g}-{snr.max():.2g}")
# What to look for: most stars sit at decent S/N with a noisy faint tail; S/N tracks BRIGHTNESS far
# more than temperature -> brightness is a nuisance we remove from the features (in 5a) but it still
# tells us where the data is trustworthy.

*Most stars sit at decent S/N with a fainter, noisier tail. **Brightness, not temperature**, sets
data quality — worth remembering later when a model struggles in some region.*

### 6c · The 110 coefficient scales at a glance  *(Your turn — Student B leads)*

In [ ]:
# GOAL: see how differently scaled the 110 (brightness-normalized) coefficients are -> WHY we z-score.
# We summarize each of the 110 columns by its mean and its spread (std) across all stars.
Xn       = normalize_by_g(X, meta["gflux"])     # the §5a (physics) coefficients
col_mean = Xn.mean(axis=0)
col_std  = Xn.std(axis=0)
idx      = np.arange(110)

fig, ax = plt.subplots()
ax.axvline(55, color="0.5", ls=":", label="BP | RP boundary")   # first 55 = BP, last 55 = RP
ax.set_xlabel("coefficient index (0-54 BP, 55-109 RP)")
ax.set_ylabel("coefficient value (mean +/- std)")
ax.set_title("The 110 coefficients live on very different scales")
ax.legend()
# <- your turn (6c): show each column's mean with its std as an error bar:
#       ax.errorbar(idx, col_mean, yerr=col_std, fmt="o", ms=3, color=COLORS["rf"], alpha=0.7)
#   Then add ax.set_yscale("symlog") to reveal the small coefficients hiding near zero.
plt.show()

print(f"column std spans {col_std.min():.2g} to {col_std.max():.2g}  (~{col_std.max()/col_std.min():.0f}x)")
# What to look for: a few LOW-index coefficients (the broad continuum modes) dominate the scale --
# exactly why standardizing (every column -> std 1) matters before any distance- or gradient-based model.

*A few coefficients are thousands of times larger than the rest. Standardizing (§5b) is exactly what
stops them from drowning out the others — this plot *is* the motivation for that step.*

### 6d · Which coefficients carry which label?  *(Your turn — Student B leads)*

In [ ]:
# GOAL: which coefficients carry information about each label? (a first feel for "what's predictive")
Xn     = normalize_by_g(X, meta["gflux"])
labels = ["Teff", "logg", "FeH"]

# correlation of every coefficient with each label -> a (110 x 3) matrix
corr = np.zeros((110, len(labels)))
for j in range(110):
    for k, lab in enumerate(labels):
        corr[j, k] = np.corrcoef(Xn[:, j], y[lab])[0, 1]

fig, ax = plt.subplots(figsize=(4.5, 7))
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_ylabel("coefficient index (0-109)")
ax.set_title("corr(coefficient, label)")
# <- your turn (6d): draw the heatmap and a colorbar:
#       im = ax.imshow(corr, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
#       fig.colorbar(im, ax=ax, label="Pearson r")
plt.show()

# Which coefficients correlate most strongly with our TARGET, [Fe/H]?
top = np.argsort(-np.abs(corr[:, labels.index("FeH")]))[:5]
print("top-5 coefficients for [Fe/H]:", top, " r =", corr[top, labels.index("FeH")].round(2))
print("strongest |r| per label:", dict(zip(labels, np.abs(corr).max(axis=0).round(2))))
# What to look for: Teff's column is BOLD (easy label); [Fe/H]'s is pale and spread across several
# coefficients -> no single "metallicity-meter", the signal is diffuse and weak.

*Temperature lights up strong correlations; metallicity's signal is faint and spread thin across
several coefficients — there is **no single "metallicity-meter."** That diffuse, weak signal is exactly
what makes [Fe/H] the hard label, and what §8's linear fit has to stitch together.*

### 6e · Label distributions and relationships  *(Your turn — Student C leads)*

In [ ]:
# GOAL: see the spread of each label and how labels relate to each other (a hand-rolled corner plot).
labels = ["Teff", "logg", "FeH"]

# Diagonal = histogram of each label; lower triangle = pairwise scatter.
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for r in range(3):
    for c in range(3):
        ax = axes[r, c]
        if r == c:
            # <- your turn (6e-i): histogram label r on the diagonal:
            #       ax.hist(y[labels[r]], bins=40, color=COLORS["nn"])
            pass
        elif r > c:
            # <- your turn (6e-ii): scatter label c (x) vs label r (y) in the lower triangle:
            #       ax.scatter(y[labels[c]], y[labels[r]], s=4, alpha=0.15, color=COLORS["nn"])
            pass
        else:
            ax.axis("off")
        if r == 2: ax.set_xlabel(PRETTY[labels[c]])
        if c == 0 and r != 0: ax.set_ylabel(PRETTY[labels[r]])
fig.suptitle("Label distributions & pairwise relationships")
plt.tight_layout(); plt.show()

print(f"metal-poor tail ([Fe/H] < -1): about {float((y['FeH'] < -1).mean()):.1%} of stars (the halo).")
# What to look for: Teff and logg show clear structure (two clumps: dwarfs vs giants); [Fe/H] is a
# broad peak near solar with a thin metal-poor tail (the halo) -> sets up the Kiel diagram next.

*Each label has its own character: a bimodal $\log g$ (a dwarf population and a giant population),
a broad near-solar [Fe/H] with a thin **metal-poor tail** (the halo). The pairwise panels already hint
at the populations we'll see in the Kiel diagram next.*

### 6f · The Kiel diagram — the astronomer's map  *(Your turn — Student C leads)*

In [ ]:
# GOAL: rebuild the astronomer's map of stellar populations straight from the labels.
# A Kiel diagram plots Teff (x) vs log g (y), colored by a third label -- by convention BOTH axes flip.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.set_xlabel(PRETTY["Teff"]); ax.set_ylabel(PRETTY["logg"])
ax.set_title("Kiel diagram")
ax.invert_xaxis()        # convention: hotter to the LEFT
ax.invert_yaxis()        # convention: dwarfs (high logg) at the BOTTOM
# <- your turn (6f): scatter Teff vs logg, colored by [Fe/H], and add a colorbar:
#       sc = ax.scatter(y["Teff"], y["logg"], c=y["FeH"], s=7, alpha=0.5, cmap="viridis")
#       fig.colorbar(sc, ax=ax, label=PRETTY["FeH"])
#   Then try coloring by "alpha" instead of "FeH": does the giant branch look chemically different?
plt.show()

# What to look for: a MAIN SEQUENCE of dwarfs (logg ~4-4.6), a GIANT BRANCH sweeping to low logg/cool
# Teff, a dense RED CLUMP near (Teff ~4800 K, logg ~2.4), and the bluest (metal-poor) points = the halo.

*Plotting just two labels recreates the Hertzsprung–Russell idea: a **main sequence** of dwarfs, a
**giant branch**, the **red clump**, and a sparse **metal-poor halo**. This $T_\mathrm{eff}$–$\log g$
plane is also where — later, in NB3/NB4 — we'll ask whether our error bars are honest **everywhere**,
not just on average.*

### 6g · Dimensionality reduction with PCA  *(Recommended — everyone)*

We have **110** coefficients per star, but they are far from independent — neighboring modes wiggle
together, and the labels only span a few physical directions. **Principal Component Analysis (PCA)**
finds new axes (linear combinations of the 110) ordered by how much of the data's variance each one
captures. Two payoffs:
1. The **cumulative explained-variance** curve tells you how many of these new axes you really need —
   i.e. how many *independent* numbers the 110 coefficients actually carry.
2. The first two components give a **2-D map** of all 30k stars at once, which we can color by a label
   to see whether the structure lines up with physics.

We run PCA on the **standardized** `normalize_by_g(X)` (physics step, then z-score so PCA isn't
dominated by the big columns from 6c). *Reference:* scikit-learn's
[PCA user guide](https://scikit-learn.org/stable/modules/decomposition.html#pca).

In [ ]:
# GOAL: how many independent directions do the 110 coefficients really carry, and what does a 2-D map look like?
Xn = normalize_by_g(X, meta["gflux"])
mu = Xn.mean(0); sd = Xn.std(0); sd = np.where(sd < 1e-12, 1.0, sd)
Xz = (Xn - mu) / sd                              # standardized (generic step) before PCA

pca    = PCA().fit(Xz)                           # all components
scores = pca.transform(Xz)                       # each star's coordinates on the new axes
cumvar = np.cumsum(pca.explained_variance_ratio_)
n90    = int(np.searchsorted(cumvar, 0.90)) + 1  # PCs needed to reach 90% of the variance

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

# Left: cumulative explained variance (WORKED for you).
ax[0].plot(np.arange(1, 111), cumvar, color=COLORS["rf"])
ax[0].axhline(0.90, color="0.5", ls="--", label="90% of variance")
ax[0].axvline(n90, color="0.5", ls=":", label=f"{n90} PCs")
ax[0].set_xlabel("number of principal components"); ax[0].set_ylabel("cumulative explained variance")
ax[0].set_title("How many directions do we need?"); ax[0].legend()

# Right: a 2-D map (PC1 vs PC2), colored by a label. Robust axis limits ignore a few outliers.
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2"); ax[1].set_title("PCA map, colored by Teff")
ax[1].set_xlim(np.percentile(scores[:, 0], [1, 99]))
ax[1].set_ylim(np.percentile(scores[:, 1], [1, 99]))
# <- your turn (6g): color the PCA map by a LABEL, then change the color label and look again:
#       sc = ax[1].scatter(scores[:, 0], scores[:, 1], c=y["Teff"], s=6, alpha=0.4, cmap="viridis")
#       fig.colorbar(sc, ax=ax[1], label=PRETTY["Teff"])
#   Re-run coloring by y["FeH"] (or y["logg"]). Which label organizes the map most cleanly?
plt.tight_layout(); plt.show()

print(f"PCs to reach 90% of variance: {n90}   (the 110 coefficients carry far fewer independent numbers)")
# What to look for: a handful of PCs already capture a big chunk of the structure; Teff sweeps smoothly
# across the map, while [Fe/H] is a subtler gradient -> the hard label, again.

*The cumulative-variance curve climbs fast at first then flattens: a handful of components already
capture a large share of the structure, and you reach 90% well before 110 — the coefficients are
**redundant**, carrying fewer independent numbers than their count suggests. The 2-D map arranges all
30k stars by their dominant variation, and a label like $T_\mathrm{eff}$ sweeps smoothly across it;
[Fe/H] is a fainter gradient — the hard label, one more time.*

### 6h · UMAP — a nonlinear map  *(Stretch — everyone, if `umap` is installed)*

PCA only finds **linear** axes. **UMAP** is a popular *nonlinear* dimensionality-reduction method: it
tries to preserve which stars are *near each other* in the full 110-D space while squashing everything
down to 2-D, so curved structure can unfold in ways PCA can't. It's slower, so we run it on a
**~3,000-star subsample**. The cell is guarded by `HAVE_UMAP` — if `umap` isn't installed it just
prints a friendly note. *Reference:* the [umap-learn docs](https://umap-learn.readthedocs.io/).

In [ ]:
# STRETCH: a nonlinear 2-D embedding of a subsample, colored by a label. Guarded + subsampled for speed.
if HAVE_UMAP:
    Xn = normalize_by_g(X, meta["gflux"])
    mu = Xn.mean(0); sd = Xn.std(0); sd = np.where(sd < 1e-12, 1.0, sd)
    Xz = (Xn - mu) / sd

    rng = np.random.default_rng(0)
    sub = rng.permutation(len(Xz))[:3000]                # ~3000 rows keeps UMAP to a few seconds
    emb = umap.UMAP(n_components=2, random_state=0).fit_transform(Xz[sub])

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=y["FeH"].to_numpy()[sub], s=8, alpha=0.6, cmap="viridis")
    fig.colorbar(sc, ax=ax, label=PRETTY["FeH"])
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.set_title("UMAP embedding (3000 stars), colored by [Fe/H]")
    plt.show()
    print("UMAP is nonlinear and slower than PCA; try coloring by y['Teff'] to compare the layouts.")
else:
    print("umap not installed -> skipping the stretch embedding (HAVE_UMAP is False).")

*UMAP often separates populations (dwarfs, giants, the metal-poor tail) into visually distinct
blobs that a linear method blurs together. It's a great *exploratory* tool — but remember the axes have
no physical units, and the layout can shift with its settings, so read it qualitatively.*

## 7 · Explore (pick one thread, ~15–20 min)

No right answer here — pick **one**, dig in, and bring what you find to the debrief.

1. **Which coefficients look most informative for [Fe/H]?** Start from the heatmap (6d). Do the
   strongest [Fe/H] coefficients sit in BP (indices 0–54) or RP (55–109)? Scatter the single best one
   against [Fe/H] and eyeball the trend.
2. **How redundant are the features?** Compute the 110×110 feature–feature correlation matrix and view
   it:
   ```python
   plt.imshow(np.corrcoef(normalize_by_g(X, meta["gflux"]).T), cmap="coolwarm", vmin=-1, vmax=1)
   ```
   Are neighboring coefficients near-duplicates? Tie this back to PCA (6g): redundancy is *why* far
   fewer than 110 components capture most of the variance.
3. **How many PCA dimensions do you really need?** From 6g, read off how many components reach 80%, 90%,
   95%. Is "the 110 coefficients carry far fewer independent numbers" fair — and how few is *few*?
4. **Where does the data thin out?** On the Kiel diagram (6f), find a sparse corner — luminous giants
   (low $\log g$), or very metal-poor stars. Roughly how many stars live there? Few examples in a
   region is the setup for **epistemic** uncertainty — a word you'll meet in NB3.

## Day 2 — a first regression, and how to *read* it

Yesterday we built intuition. Today we do the simplest possible prediction — and then spend most of our
time **interpreting** it. (We are *not* chasing accuracy or worrying about overfitting yet; the proper
train/calibration/test discipline and overfitting are NB2's job.)

## 8 · A first linear fit for [Fe/H]  *(Worked)*

We've stared at the data enough to suspect [Fe/H] is **hard** — its signal is faint and spread across
many coefficients. But "hard" isn't "hopeless." Let's do the simplest thing: a **linear regression**
that predicts [Fe/H] as a *weighted sum* of the 110 preprocessed coefficients, on a plain hold-out
split. No tuning — just a sanity check that prediction is *possible*.

Two numbers to read the result by:
- **RMSE** (root-mean-square error): the typical size of the miss, in the label's units (here dex).
  Smaller is better.
- **R²**: the fraction of the label's variance the model explains. 1.0 is perfect; 0 is no better than
  always guessing the average.

> *We keep this deliberately bare:* one fit, one held-out test set. The real machinery —
> train/calibration/test done right, cross-validation, overfitting, regularization — arrives in
> **NB2**. This number is just the **baseline** every NB2 model will be measured against.

In [ ]:
# Preprocess with the §5 recipe (physics normalize -> split -> standardize on train only).
Xn = normalize_by_g(X, meta["gflux"])
sp = train_cal_test_split(Xn, y, seed=0)         # same fixed-seed split; we use just train + test today
Xtr_s, Xte_s = standardize(sp["X_train"], sp["X_test"])

y_tr = sp["y_train"]["FeH"].to_numpy()           # PRIMARY TARGET = [Fe/H]
y_te = sp["y_test"]["FeH"].to_numpy()

lin  = LinearRegression().fit(Xtr_s, y_tr)       # fit on train
pred = lin.predict(Xte_s)                        # predict on held-out test

m = regression_metrics(y_te, pred)
print({k: round(v, 3) for k, v in m.items()})    # RMSE in dex, plus MAE, bias, R2

ax = plot_pred_vs_true(y_te, pred, title="Linear baseline: [Fe/H]", units=" dex")
plt.show()

*A plain linear model already predicts [Fe/H] to a typical error of **a few tenths of a dex**, explaining
**over half** the variance — proof the signal is real, even if faint. The scatter around the 1:1 line
is what every fancier model in NB2 will try to shrink. **Remember this number: it's the baseline to
beat.***

## 9 · Interpret the fit — what is the model actually *using*?

A prediction you can't explain is hard to trust. The rest of Day 2 is about **reading** this linear
model: which coefficients matter, how much each one contributes for a given star, and how this connects
to a tool you may have heard of (**SHAP**). Interpretation is where a linear model shines — every step
here is exact arithmetic, no black box.

### 9a · Coefficients: raw vs standardized  *(Worked)*

A linear model is $\hat{y} = \beta_0 + \sum_i \beta_i x_i$ — each feature $x_i$ gets a weight
$\beta_i$. It's tempting to read $|\beta_i|$ as "feature $i$'s importance," but that only works if the
features share a scale. On the **raw** `normalize_by_g(X)` the columns span a thousand-fold range
(remember 6c), so the raw $\beta_i$ are on wildly different footings and tell you almost nothing. On
the **standardized** features (every column z-scored to spread 1), a one-unit change means "one standard
deviation" *for every feature*, so $|\beta_i|$ becomes comparable across features — a fair read of each
coefficient's influence. Let's fit both and compare.

In [ ]:
# Fit the SAME linear model on the RAW (G-normalized) features and on the STANDARDIZED features.
Xtr_raw = sp["X_train"]                           # normalize_by_g, NOT z-scored
lin_raw = LinearRegression().fit(Xtr_raw, y_tr)
lin_std = LinearRegression().fit(Xtr_s,  y_tr)    # z-scored (from §8)

print("raw-feature |coef| range:          %.3g  to  %.3g" % (np.abs(lin_raw.coef_).min(), np.abs(lin_raw.coef_).max()))
print("standardized-feature |coef| range: %.3g  to  %.3g" % (np.abs(lin_std.coef_).min(), np.abs(lin_std.coef_).max()))

# Plot the STANDARDIZED coefficients as a stem over the 110 features.
fig, ax = plt.subplots(figsize=(11, 4))
ax.stem(np.arange(110), lin_std.coef_, basefmt=" ")
ax.axvline(55, color="0.5", ls=":", label="BP | RP boundary")
ax.set_xlabel("coefficient index (0-54 BP, 55-109 RP)")
ax.set_ylabel(r"standardized coefficient $\beta_i$")
ax.set_title("Standardized linear coefficients for [Fe/H]")
ax.legend()
plt.show()

top = np.argsort(-np.abs(lin_std.coef_))[:8]
print("top-8 most influential coefficients (by |standardized beta|):", top)
print("their betas:", lin_std.coef_[top].round(3))

*The raw coefficients span a huge range (driven by the feature scales, not by importance), so they're
**uninterpretable**. The standardized coefficients are on a common footing: a handful of them stand out
with the largest $|\beta_i|$, and *those* are the coefficients the model leans on hardest for [Fe/H].
Note these top coefficients are mostly **low-index continuum modes** — the same broad-shape modes that
dominated 6c — telling us a lot of [Fe/H] information rides on the overall spectral shape.*

### 9b · Effect sizes for one star  *(Your turn)*

A big coefficient $\beta_i$ is a *sensitivity* (a **gradient**): "if coefficient $i$ moved, the
prediction would move this much." But what actually drives **one particular star's** prediction is the
**effect** $\text{effect}_i = \beta_i \, x_i$ — the coefficient *times that star's actual value*. A
feature with a big weight but a near-average value contributes little; a feature with a modest weight
but an extreme value can dominate. Summing the effects (plus the intercept) reproduces the prediction
exactly. The **fractional contribution** $|\text{effect}_i| / \sum_j |\text{effect}_j|$ then tells you
which handful of coefficients drive *this* star.

In [ ]:
# GOAL: for ONE star, find which coefficients actually drive its [Fe/H] prediction.
beta = lin_std.coef_                              # standardized coefficients from 9a
s    = 0                                          # which test star to dissect (try other indices!)
xi   = Xte_s[s]                                   # this star's standardized coefficients

# effect_i = beta_i * x_i  (the actual contribution of coefficient i for THIS star)
effect = beta * xi
# fractional contribution = |effect_i| / sum_j |effect_j|
frac   = np.abs(effect) / np.abs(effect).sum()

order = np.argsort(-frac)[:10]                    # the 10 biggest contributors
fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlabel("coefficient index"); ax.set_ylabel("signed effect  (beta * x)")
ax.set_title(f"Top contributors to star #{s}'s [Fe/H] prediction")
# <- your turn (9b): bar-plot the signed EFFECTS of the top contributors:
#       ax.bar(np.arange(10), effect[order], color=COLORS["rf"])
#       ax.set_xticks(np.arange(10)); ax.set_xticklabels(order, fontsize=8)
plt.show()

print(f"star #{s}: predicted [Fe/H] = {lin_std.predict(xi[None])[0]:+.3f}, true = {y_te[s]:+.3f}")
print("top-5 coefficients by fractional contribution:", order[:5])
print("their fractional contributions:", frac[order[:5]].round(3))
# What to look for: only a HANDFUL of coefficients carry most of the prediction for this star. Compare
# 'order' here (the EFFECT for one star) to 9a's top-|beta| list (the GRADIENT, same for all stars):
# they overlap but are NOT identical -- gradient = sensitivity, effect = this star's actual contribution.

*Only a handful of coefficients carry most of this star's prediction. The effect ranking overlaps
the 9a gradient ranking but isn't identical — which is exactly the distinction: $\beta_i$ alone is a
**sensitivity** (the same for every star), while $\beta_i x_i$ is the **actual contribution** for
*this* star.*

### 9c · Stretch: SHAP-for-linear, and an optional OLS summary

You may have heard of **SHAP** (SHapley Additive exPlanations) — a popular, model-agnostic way to
attribute a prediction to its features. For a **linear** model SHAP has a beautiful closed form: the
SHAP value of feature $i$ is
$$\phi_i = \beta_i \,(x_i - \bar{x}_i),$$
i.e. the coefficient times *how far this star's feature sits from the average star*. Because we
standardized on the training set, the training mean $\bar{x}_i \approx 0$, so $\phi_i \approx \beta_i
x_i$ — **exactly the effect size from 9b**. So 9b *was* SHAP-for-linear all along. We verify the
identity below by hand, **and** (guarded by `HAVE_SHAP`) cross-check it against the real
[`shap`](https://shap.readthedocs.io/) library's `LinearExplainer` — they should agree to ~$10^{-12}$.

The general (nonlinear) case has *no* such closed form, which is exactly where the `shap` library earns
its keep: it estimates these additive attributions for any model. `shap` **is installed** in this
environment, and **you'll use it for real in NB2** on your *nonlinear* model (KNN / random forest /
neural net), where the hand-computed linear trick no longer applies. For the theory, Christoph Molnar's
[*Interpretable Machine Learning*](https://christophm.github.io/interpretable-ml-book/) book is the
standard reference. As a second, optional extra, `statsmodels` (guarded by `HAVE_SM`) gives a full
**OLS summary** with **p-values** and **confidence intervals** for each coefficient.

In [ ]:
# (1) SHAP-for-linear, by hand: phi_i = beta_i * (x_i - mean_i).  Show it equals 9b's effect.
beta     = lin_std.coef_
mean_tr  = Xtr_s.mean(axis=0)                     # training-set means (~0 because we standardized on train)
s        = 0
xi       = Xte_s[s]

phi    = beta * (xi - mean_tr)                    # SHAP-for-linear
effect = beta * xi                               # 9b effect size
print(f"max |phi - effect| over 110 features = {np.abs(phi - effect).max():.2e}  (≈ 0 -> identical)")
print(f"sum(phi) + base = {phi.sum() + (lin_std.intercept_ + beta @ mean_tr):+.3f}"
      f"   prediction = {lin_std.predict(xi[None])[0]:+.3f}")

# (1b) Cross-check against the REAL shap library (instant for a linear model via LinearExplainer).
#      shap may print a benign "Subsampling ... samples" note -> harmless. We pass the FULL training
#      background so its reference mean matches our hand-computed mean_tr exactly.
if HAVE_SHAP:
    masker      = shap.maskers.Independent(Xtr_s, max_samples=Xtr_s.shape[0])
    explainer   = shap.LinearExplainer(lin_std, masker)
    shap_star   = np.asarray(explainer(xi[None]).values).reshape(-1)   # (110,) for this one star
    print(f"max |shap - hand| over 110 features = {np.abs(shap_star - phi).max():.2e}"
          f"  (shap's LinearExplainer reproduces our hand-computed values)")
else:
    print("shap not installed -> skipping the library cross-check (HAVE_SHAP is False).")

# (2) Optional: a full OLS summary with p-values / confidence intervals (if statsmodels is available).
if HAVE_SM:
    Xsm = sm.add_constant(Xtr_s)                  # add the intercept column
    ols = sm.OLS(y_tr, Xsm).fit()
    top = np.argsort(-np.abs(beta))[:5]           # the 5 most influential coefficients from 9a
    ci  = ols.conf_int()                          # (n+1, 2): rows are [const, feat0, feat1, ...]
    print("\ntop-5 coefficients with statsmodels p-values & 95% CIs:")
    for j in top:
        lo, hi = ci[j + 1]                        # +1 to skip the const row
        print(f"  coef {j:3d}:  beta = {ols.params[j+1]:+.3f}   p = {ols.pvalues[j+1]:.1e}   "
              f"95% CI [{lo:+.3f}, {hi:+.3f}]")
    print(f"\n(OLS R^2 on TRAIN = {ols.rsquared:.3f}; slightly above the §8 hold-out R^2, as expected.)")
else:
    print("\nstatsmodels not installed -> skipping the OLS summary (HAVE_SM is False).")

*The hand-computed SHAP values match 9b's effects to machine precision, and the real `shap` library's
`LinearExplainer` reproduces them to ~$10^{-12}$ — for a linear model, "SHAP" really is just
$\beta_i (x_i - \bar{x}_i)$. The win is that the **same library** carries over to your *nonlinear* model
in NB2, where no such hand formula exists. The `statsmodels` summary adds inferential detail (p-values,
confidence intervals) that says **which coefficients are statistically distinguishable from zero** —
handy, though with 110 correlated features it should be read with care.*

### 9d · A residual teaser — the errors aren't uniform  *(Worked)*

One last look before we close. A single RMSE hides *where* the model errs. Let's split the test
residuals (predicted − true) by **metallicity** and by **surface gravity** and compare the spread. This
isn't a failure to fix — it's **interesting structure** we'll quantify properly in NB3.

In [ ]:
resid   = pred - y_te                            # signed error in dex (pred & y_te from §8)
feh_te  = sp["y_test"]["FeH"].to_numpy()
logg_te = sp["y_test"]["logg"].to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
plot_residuals(y_te, pred, feature=feh_te,  feature_name=PRETTY["FeH"],  ax=ax[0]); ax[0].set_title("residuals vs [Fe/H]")
plot_residuals(y_te, pred, feature=logg_te, feature_name=PRETTY["logg"], ax=ax[1]); ax[1].set_title("residuals vs log g")
plt.tight_layout(); plt.show()

# Quantify the spread in the hard vs easy regions:
mpoor = feh_te < -0.7;  mrich = feh_te > -0.3     # metal-poor vs metal-rich
giant = logg_te < 2.5;  dwarf = logg_te > 4.0     # luminous giants vs compact dwarfs
print(f"residual std by [Fe/H]:  metal-poor {resid[mpoor].std():.3f} dex   metal-rich {resid[mrich].std():.3f} dex")
print(f"residual std by log g:   giants     {resid[giant].std():.3f} dex   dwarfs     {resid[dwarf].std():.3f} dex")

*The errors are **not** uniform: the residual band fans out toward **metal-poor stars** (weak metal
lines → little information) and toward **luminous giants** (sparse, most-extrapolated). Metal-rich
dwarfs are the easy case. This uneven, region-dependent error is the single observation we'll pull on
for the rest of the project — and it's why honest uncertainty has to be **bigger where the data is less
informative**. (Note it's metal-poor stars and giants that are hard here — *not* hot stars.)*

## 10 · Wrap-up, and the bridge to NB2

**What you now know:**
- The **inputs** are 110 compressed-spectrum coefficients per star; the **labels** are
  $T_\mathrm{eff}$ (easy), $\log g$ (medium), and **[Fe/H]** (hard — our target).
- There are **two different "normalizations"**: a **physics** one (divide by G — removes
  brightness/distance, keeps shape) and a **generic** one (z-score — a numerical convenience so every
  coefficient gets a fair say). Don't conflate them.
- The data is **redundant** (PCA: far fewer than 110 independent directions) and its quality tracks
  **brightness**, not temperature.
- A bare linear fit already predicts [Fe/H] to a few tenths of a dex, and you can **read** it: which
  coefficients matter (standardized $\beta$), what drives one star (effect sizes = SHAP-for-linear).
- Its errors are **uneven** — bigger for metal-poor stars and luminous giants.

**Debrief checklist:** bring your **2–3 annotated plots**, and as a team make sure A/B/C between you
have covered the whole EDA menu (6a–6h).

**Next — NB2.** We build a *proper* pipeline **early**: the train / calibration / test split done
right, scikit-learn `Pipeline`, no leakage, and the overfitting / cross-validation discipline we
skipped today. Then each of you swaps in your own model — **A: KNN · B: Random Forest · C: Neural
network** — and tries to beat today's linear baseline. The calibration split gets held back for the
**uncertainty** work in NB3/NB4.

> One hook to carry with you: today we *noticed* that errors grow for metal-poor stars and giants. In a
> couple of weeks you'll turn that observation into a **guaranteed** error bar — and find exactly where
> the guarantee strains.

### Further reading
- **The paper:** Laroche & Speagle (2025), [arXiv:2404.07316](https://arxiv.org/abs/2404.07316)
  ([ADS](https://ui.adsabs.harvard.edu/abs/2025ApJ...979....5L)) — the label-transfer method this
  project is based on.
- **Preprocessing & scaling:** scikit-learn
  [preprocessing user guide](https://scikit-learn.org/stable/modules/preprocessing.html).
- **PCA:** scikit-learn [decomposition / PCA user guide](https://scikit-learn.org/stable/modules/decomposition.html#pca).
- **UMAP:** the [umap-learn docs](https://umap-learn.readthedocs.io/).
- **Interpretation:** Christoph Molnar, [*Interpretable Machine Learning*](https://christophm.github.io/interpretable-ml-book/)
  (free online); and the [`shap` library docs](https://shap.readthedocs.io/) — used hands-on in NB2 for
  the general (nonlinear) case.
- **The astronomy:** Gaia BP/RP spectra ([ESA overview](https://www.cosmos.esa.int/web/gaia/iow_20220131));
  APOGEE / ASPCAP labels ([SDSS](https://www.sdss4.org/dr17/irspec/)); the Hertzsprung–Russell / Kiel
  diagram ([Wikipedia](https://en.wikipedia.org/wiki/Hertzsprung%E2%80%93Russell_diagram)).